In [ ]:
from sklearn import datasets
import pandas as pd
from scipy.stats import zscore
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier

# Armamento del Data Frame proveniente de Scikit-Learn

data = datasets.load_wine()

df = pd.DataFrame(data["data"],columns=data["feature_names"])
df["class"] = data["target"]

df

In [ ]:
# Minería de datos atípicos
df_zcore = df.copy()
outliers = []

print("Outliers \n")
for c in df.columns[:-1]:
    df_zcore[f"{c}_zscore"] = zscore(df_zcore[c]).abs()
    outliers = df_zcore.loc[df_zcore[f"{c}_zscore"] > 3,c]
    print(f"columna '{c}': {outliers.shape[0]} ")

In [ ]:
# Figura múltiple sobre promedios
fig = make_subplots(rows=4, cols=3)

column = 1
row = 1
for c in df.columns[1:-1]:
    df_mean = df.groupby("class")[c].mean().reset_index()
    fig.add_trace(go.Bar(x=df_mean["class"], y=df_mean[c], name=f"{c} mean"),row=row, col=column)
    column += 1
    if column == 4:
        row += 1
        column = 1

fig.update_layout(height=1300, width=850)
fig

##### Como vimos en el ejemplo 1 la estimación en la función objetivo tiene un cierto nivel de variabilidad, este meta-algoritmo reduce esta última y además meta-estima la función 

In [ ]:
# Generando modelo de clasficación

model_forest = RandomForestClassifier(n_estimators=100, # 100 Árboles de decisión
                                    criterion="gini", # Impureza gini
                                    max_features="sqrt", # Raíz del número de característicad
                                    bootstrap=True, # Boostrapping (Muestreo)
                                    max_samples=0.75, # Muestras del 75% para el entrenamiento
                                    oob_score=True) # Evaluación de muestras fuera de la bolsa

model_forest.fit(df[df.columns[:-1]], df["class"])

print(f"Accuracy: {model_forest.oob_score_}")